In [0]:
from pyspark.sql.functions import *

In [0]:
#workspace.`0_bronze`.cust_data_tbl
def str_Col_Trim( df):
    for field in df.schema.fields:
        if field.dataType == StringType():
            df = df.withColumn(field.name, trim(col(field.name)))
    return df

def Transform_cust(df):
    #basic
    df=str_Col_Trim(df)
    #customised
    df=(
         df.withColumn("cleaned_cust_id",regexp_extract(col("cust_id"),"(\\d+)",1).cast("int"))
            .dropDuplicates(["cleaned_cust_id"])
            .withColumn("ingest_date",col("ingest_ts").cast("date") )
            .drop("ingest_ts")
            .drop("cust_id")
            .withColumnRenamed("cleaned_cust_id","cust_id")
        )
    df=df.select("cust_id","Cust_Name","ingest_date")
    return df


In [0]:
 ########################################################33
df=(spark.table("workspace.`0_bronze`.cust_data_tbl"))
df.printSchema()
df= Transform_cust(df) #-->Trim-->Clean Custid-->Dedup-->cast
df.printSchema()
   
df.display()
#df.write.mode("overwrite").saveAsTable("workspace.`1_Silver`.cust_data_tbl")

In [0]:
df.createOrReplaceTempView("source_view")

spark.sql("""
  MERGE INTO workspace.`1_Silver`.cust_data_tbl AS target
  USING source_view AS source
  ON target.cust_id = source.cust_id
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
%skip
%sql
show tables;
select *
from workspace.`1_Silver`.cust_data_tbl ;
--workspace.`1_silver`.cust_data_tbl
DROP TABLE IF EXISTS `1_Silver`.cust_data_tbl